In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS pipeline_1.gold


In [0]:
SILVER_TABLE = "pipeline_1.silver.silver_events"
CHECKPOINT_PATH = "/Volumes/pipeline_1/bronze/checkpoints_volume/gold_checkpoint"

GOLD_AVG = "pipeline_1.gold.metrics_avg"
GOLD_MIN = "pipeline_1.gold.metrics_min"
GOLD_MAX = "pipeline_1.gold.metrics_max"
GOLD_STDDEV = "pipeline_1.gold.metrics_stddev"

In [0]:
from pyspark.sql.types import NumericType

silver_schema = spark.table(SILVER_TABLE).schema

# Определяем колонки с числовыми данными
numeric_cols = [
    field.name for field in silver_schema
    if isinstance(field.dataType, NumericType)
    and field.name not in ("device_id",)
]


In [0]:
df_silver = (
    spark.readStream
    .table(SILVER_TABLE)
)

In [0]:
from pyspark.sql.functions import window, avg, min, max, stddev, current_timestamp, col
from delta.tables import DeltaTable

def write_to_gold(microBatchDF, batchId):
    from pyspark.sql.functions import window, avg, min, max, stddev, current_timestamp, col
    from delta.tables import DeltaTable
    
    # Динамически создаем список агрегаций для каждой числовой колонки 
    agg_exprs_avg = [avg(c).alias(c) for c in numeric_cols]
    agg_exprs_min = [min(c).alias(c) for c in numeric_cols]
    agg_exprs_max = [max(c).alias(c) for c in numeric_cols]
    agg_exprs_stddev = [stddev(c).alias(c) for c in numeric_cols]

    # Цикл по 4 агрегациям - для каждой свой список и своя таблица
    for agg_exprs, table_name in [
        (agg_exprs_avg, GOLD_AVG),
        (agg_exprs_min, GOLD_MIN),
        (agg_exprs_max, GOLD_MAX),
        (agg_exprs_stddev, GOLD_STDDEV)
    ]:
        # Группируем по device_id и временному окну в 1 минуту, считаем агрегации, добавляем колонки с временем и временем записи
        agg_df = (
            microBatchDF
            .groupBy("device_id", window("event_time", "1 minute"))
            .agg(*agg_exprs)
            .withColumn("window_start", col("window.start"))
            .withColumn("window_end", col("window.end"))
            .withColumn("inserted_at", current_timestamp())
            .drop("window")
        )

        #  Если таблицы еще нет, то создаем ее, иначе делаем merge по device_id и временному окну
        # Таким образом решаем проблему запаздывающих данных (если за ранее записанную минуту пришли новые данные - перепишем актуальные)

        if not spark.catalog.tableExists(table_name):
            agg_df.write.format("delta").saveAsTable(table_name)
        else:
            delta_table = DeltaTable.forName(spark, table_name)
            (delta_table.alias("target")
                .merge(
                    agg_df.alias("source"),
                    "target.device_id = source.device_id AND target.window_start = source.window_start"
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute())


# Запускаем запись из силвер таблицы, для каждого нового батча вызываем функцию write_to_gold
(df_silver
    .writeStream
    .foreachBatch(write_to_gold)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start())